# DeepWiki Required Infrastructure Deployment

This notebook deploys the **must-have** infrastructure resources for DeepWiki.

## Resources Deployed
1. **User Managed Identity** - Authentication for all Azure services
2. **Azure Storage Account** - Blob storage for wiki cache and data
3. **Azure OpenAI** - LLM and embedding models
4. **Application Insights + Log Analytics** - Monitoring and diagnostics
5. **Azure App Service** - Container hosting (Linux, with network restrictions)
6. **Role Assignments** - RBAC permissions for managed identity
7. **Network Security Perimeter (NSP)** - Optional network security

## Prerequisites
- Azure CLI authenticated (`az login`)
- Python 3.10+ with required packages
- Configuration completed in `config.py`

## Deployment Order
Run cells in sequence - each block depends on previous deployments.

## Block 1: Setup & Configuration

Load configuration and authenticate with Azure.

In [1]:
# Import required libraries
import json
import importlib
from pathlib import Path
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.resource.resources.models import ResourceGroup

# Import deployment utilities (includes parameter template resolver and resource checks)
# Reload to ensure latest changes are picked up
import utils
importlib.reload(utils)
from utils import deploy_arm, load_config, get_parameters_for_deployment, check_and_prepare_deployment

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Load configuration from config.py using the utility function
config = load_config("config.py")

print("✓ Configuration loaded successfully")
print(f"  - Subscription ID: {config['subscription_id']}")
print(f"  - Resource Group: {config['resource_group']}")
print(f"  - Location: {config['location']}")
print(f"  - Managed Identity: {config['dri_copilot_identity_name']}")
print(f"  - Resource Tags: {config['resource_tags']}")

✓ Configuration loaded successfully
  - Subscription ID: 4f3f8f41-5643-4664-8c12-ce6b78ceb81f
  - Resource Group: RG-ORCAS-DEEPWIKI
  - Location: eastasia
  - Managed Identity: mid-orcas-deepwiki
  - Resource Tags: {'Owner': 'DaP CN Orcas'}


In [3]:
# Authenticate with Azure using Azure CLI credentials
credential = AzureCliCredential()

# Test authentication
credential.get_token("https://management.azure.com/.default")

# Initialize resource management client
resource_client = ResourceManagementClient(credential, config['subscription_id'])

print("✓ Azure authentication successful")

✓ Azure authentication successful


## Block 2: Create Resource Group

Create or verify the Azure Resource Group for DeepWiki resources.

In [ ]:
# Create Resource Group
resource_group_name = config['resource_group']
location = config['location']

print(f"Creating resource group: {resource_group_name}")
print(f"Location: {location}")

try:
    # Check if resource group exists
    existing_rg = resource_client.resource_groups.get(resource_group_name)
    print(f"✓ Resource group '{resource_group_name}' already exists")
    print(f"  - Location: {existing_rg.location}")
    print(f"  - Provisioning State: {existing_rg.properties.provisioning_state}")
except Exception:
    # Create resource group
    print(f"Creating resource group '{resource_group_name}'...")
    resource_group_params = ResourceGroup(location=location, tags=config.get('resource_tags', {}))
    result = resource_client.resource_groups.create_or_update(
        resource_group_name,
        resource_group_params
    )
    print(f"✓ Resource group created successfully")
    print(f"  - Location: {result.location}")
    print(f"  - Provisioning State: {result.properties.provisioning_state}")

## Block 3: Deploy User Managed Identity

Deploy the User-Assigned Managed Identity that will be used by all services.

In [ ]:
print("=" * 60)
print("DEPLOYING: User Managed Identity")
print("=" * 60)
print(f"Identity Name: {config['dri_copilot_identity_name']}")
print(f"Location: {config['location']}")

# Check if resource exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['dri_copilot_identity_name'],
    resource_type="managed_identity",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    managed_identity_params = get_parameters_for_deployment("UMI", config)
    
    # Deploy
    managed_identity_outputs = deploy_arm(
        template_file_path="templates/UMI.Template.json",
        deployment_name="deepwiki-managed-identity",
        parameters=managed_identity_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    # Store outputs for later use
    managed_identity_principal_id = managed_identity_outputs.get('managedIdentityPrincipalId', {}).get('value', '')
    managed_identity_client_id = managed_identity_outputs.get('managedIdentityClientId', {}).get('value', '')
    
    print(f"\n✓ Managed Identity deployed successfully")
    print(f"  - Principal ID: {managed_identity_principal_id}")
    print(f"  - Client ID: {managed_identity_client_id}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 4: Deploy Azure Storage Account

Deploy the Storage Account for blob storage (wiki cache, embeddings, etc.).

In [ ]:
print("=" * 60)
print("DEPLOYING: Azure Storage Account")
print("=" * 60)
print(f"Storage Account: {config['storage_account_name']}")
print(f"Location: {config['location']}")

# Check if resource exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['storage_account_name'],
    resource_type="storage",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    storage_params = get_parameters_for_deployment("STORAGE", config)
    
    # Deploy
    storage_outputs = deploy_arm(
        template_file_path="templates/STORAGE.Template.json",
        deployment_name="deepwiki-storage",
        parameters=storage_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    print(f"\n✓ Storage Account deployed successfully")
    if storage_outputs:
        print(f"  - Blob Endpoint: {storage_outputs.get('primaryBlobEndpoint', {}).get('value', 'N/A')}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 5: Deploy Azure OpenAI Service

Deploy Azure OpenAI service with model deployments.

**Note:** Run this block only if `is_creating_open_ai_endpoint = True` in config.py

In [ ]:
if config.get('is_creating_open_ai_endpoint', False):
    print("=" * 60)
    print("DEPLOYING: Azure OpenAI Service")
    print("=" * 60)
    print(f"Resource Name: {config['open_ai_resource_name']}")
    print(f"Location: {config['location']}")
    print(f"Reasoning Model: {config['open_ai_reasoning_model_model_name']}")
    print(f"Embedding Model: {config['open_ai_embedding_model_name']}")
    
    # Check if resource exists and handle location mismatch
    should_deploy, effective_location, message = check_and_prepare_deployment(
        resource_name=config['open_ai_resource_name'],
        resource_type="openai",
        config_location=config['location'],
        subscription_id=config['subscription_id'],
        resource_group=config.get('open_ai_resource_group', '') or config['resource_group'],
        credential=credential
    )
    print(f"\n{message}")
    
    if should_deploy:
        # Load parameters from template (resolved from config.py)
        aoai_params = get_parameters_for_deployment("AOAI", config)
        
        # Deploy
        aoai_outputs = deploy_arm(
            template_file_path="templates/AOAI.Template.json",
            deployment_name="deepwiki-aoai",
            parameters=aoai_params,
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group']
        )
        
        print(f"\n✓ Azure OpenAI deployed successfully")
    else:
        print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
        print(f"  Please resolve the issue above before continuing.")
else:
    print("=" * 60)
    print("SKIPPING: Azure OpenAI Service")
    print("=" * 60)
    print(f"Reason: is_creating_open_ai_endpoint = False in config.py")
    print(f"Using existing Azure OpenAI resource: {config['open_ai_resource_name']}")

## Block 6: Deploy Application Insights & Log Analytics

Deploy Application Insights with Log Analytics workspace for monitoring.

In [ ]:
print("=" * 60)
print("DEPLOYING: Application Insights & Log Analytics")
print("=" * 60)
print(f"Application Insights: {config['application_insights_name']}")
print(f"Log Analytics: {config['log_analytics_workspace_name']}")
print(f"Location: {config['location']}")

# Check if Application Insights exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['application_insights_name'],
    resource_type="appinsights",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    appinsights_params = get_parameters_for_deployment("APPINSIGHT", config)
    
    # Deploy
    appinsights_outputs = deploy_arm(
        template_file_path="templates/APPINSIGHT.Template.json",
        deployment_name="deepwiki-appinsights",
        parameters=appinsights_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    # Store connection string for App Service deployment
    appinsights_connection_string = appinsights_outputs.get('applicationInsightsConnectionString', {}).get('value', '')
    
    print(f"\n✓ Application Insights deployed successfully")
    if appinsights_connection_string:
        print(f"  - Connection String: {appinsights_connection_string[:50]}...")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 6.5: Deploy Azure Container Registry

Deploy Azure Container Registry for storing the DeepWiki container image.
The Managed Identity will be granted AcrPull role for pulling images.

In [4]:
print("=" * 60)
print("DEPLOYING: Azure Container Registry")
print("=" * 60)
print(f"Container Registry: {config['container_registry_name']}")
print(f"Location: {config['location']}")

# Check if ACR exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['container_registry_name'],
    resource_type="acr",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    acr_params = get_parameters_for_deployment("ACR", config)
    
    # Deploy
    acr_outputs = deploy_arm(
        template_file_path="templates/ACR.Template.json",
        deployment_name="deepwiki-acr",
        parameters=acr_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    print(f"\n✓ Container Registry deployed successfully")
    if acr_outputs:
        print(f"  - Login Server: {acr_outputs.get('containerRegistryLoginServer', {}).get('value', 'N/A')}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

DEPLOYING: Azure Container Registry
Container Registry: acrorcascodewiki
Location: eastasia

⚠ Resource already exists in 'eastasia'. Will UPDATE existing resource.
Starting deployment: deepwiki-acr-202601151227
This may take several minutes...
✓ Deployment successful!

✓ Container Registry deployed successfully
  - Login Server: acrorcascodewiki.azurecr.io


## Block 7: Deploy Azure App Service

Deploy App Service (Linux container mode) with container from Azure Container Registry.

**Container Configuration:**
- Uses custom Docker image from ACR (built via `publish-web.ps1`)
- Managed Identity for ACR authentication
- Image: `{container_registry_name}.azurecr.io/{container_image_name}:latest`

**Network Restrictions Applied:**
- Allow: AzureTrafficManager (priority 200)
- Allow: CorpNetPublic (priority 400)
- Allow: CorpNetSAW (priority 401)
- Deny: All other traffic

In [5]:
print("=" * 60)
print("DEPLOYING: Azure App Service")
print("=" * 60)
print(f"App Service Plan: {config['app_service_plan_name']}")
print(f"App Service: {config['app_service_name']}")
print(f"SKU: {config['app_service_plan_sku']}")
print(f"Container Registry: {config['container_registry_name']}")
print(f"Container Image: {config['container_image_name']}:latest")
print(f"Network Restrictions: {config['enable_app_service_network_restrictions']}")

# Check if App Service exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['app_service_name'],
    resource_type="app_service",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    appservice_params = get_parameters_for_deployment("WEB", config)
    
    # Override applicationInsightsConnectionString if we have it from previous deployment
    if 'appinsights_connection_string' in globals() and appinsights_connection_string:
        appservice_params['applicationInsightsConnectionString'] = appinsights_connection_string
    
    # Deploy
    appservice_outputs = deploy_arm(
        template_file_path="templates/WEB.Template.json",
        deployment_name="deepwiki-appservice",
        parameters=appservice_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    print(f"\n✓ App Service deployed successfully")
    if appservice_outputs:
        print(f"  - URL: {appservice_outputs.get('appServiceUrl', {}).get('value', 'N/A')}")
        print(f"  - Container: {appservice_outputs.get('containerImageFull', {}).get('value', 'N/A')}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

DEPLOYING: Azure App Service
App Service Plan: ASP-codewiki-orcas
App Service: orcascodewiki
SKU: P2mv4
Container Registry: acrorcascodewiki
Container Image: deepwiki:latest
Network Restrictions: True

Resource does not exist. Will create in 'eastasia'.
Starting deployment: deepwiki-appservice-202601151228
This may take several minutes...
✓ Deployment successful!

✓ App Service deployed successfully
  - URL: https://orcascodewiki.azurewebsites.net
  - Container: acrorcascodewiki.azurecr.io/deepwiki:latest


## Block 8: Configure Role Assignments

Assign RBAC roles to the Managed Identity:
- **Cognitive Services OpenAI User** on Azure OpenAI
- **Storage Blob Data Contributor** on Storage Account
- **Monitoring Metrics Publisher** on Application Insights

In [ ]:
print("=" * 60)
print("DEPLOYING: Role Assignments")
print("=" * 60)

# Ensure we have Azure credential (in case kernel was restarted)
from azure.identity import AzureCliCredential
from azure.mgmt.msi import ManagedServiceIdentityClient

try:
    credential
except NameError:
    credential = AzureCliCredential()
    print("✓ Azure credential initialized")

# Get managed identity principal ID from previous deployment or retrieve from Azure
_has_principal_id = 'managed_identity_principal_id' in globals() and globals().get('managed_identity_principal_id')

if not _has_principal_id:
    print("Retrieving Managed Identity info from Azure...")
    msi_client = ManagedServiceIdentityClient(credential, config['subscription_id'])
    identity = msi_client.user_assigned_identities.get(
        config['resource_group'],
        config['dri_copilot_identity_name']
    )
    managed_identity_principal_id = identity.principal_id
    managed_identity_client_id = identity.client_id
    print(f"  - Principal ID: {managed_identity_principal_id}")
    print(f"  - Client ID: {managed_identity_client_id}")
else:
    print(f"Using Managed Identity from previous deployment:")
    print(f"  - Principal ID: {managed_identity_principal_id}")

# Determine Azure OpenAI resource group (may be different from main resource group)
open_ai_rg = config.get('open_ai_resource_group', '') or ''
if open_ai_rg:
    print(f"\n📍 Azure OpenAI in external resource group: {open_ai_rg}")
else:
    print(f"\n📍 Azure OpenAI in same resource group: {config['resource_group']}")

# Load parameters from template (resolved from config.py)
role_params = get_parameters_for_deployment("RBAC", config)

# Override managedIdentityPrincipalId with the actual value from deployment
role_params['managedIdentityPrincipalId'] = managed_identity_principal_id

# Clear deploymentIdentityPrincipalId to avoid org policy violation (SFI blocks User RBAC)
role_params['deploymentIdentityPrincipalId'] = ""

print(f"\nAssigning roles to Managed Identity: {config['dri_copilot_identity_name']}")
print(f"  - Azure OpenAI ({config['open_ai_resource_name']}): Cognitive Services OpenAI User")
print(f"  - Storage ({config['storage_account_name']}): Storage Blob Data Contributor")
print(f"  - App Insights ({config['application_insights_name']}): Monitoring Metrics Publisher")
print(f"\n⚠ Note: User role assignments skipped (blocked by org SFI policy)")

# Deploy role assignments
role_outputs = deploy_arm(
    template_file_path="templates/RBAC.Template.json",
    deployment_name="deepwiki-role-assignments",
    parameters=role_params,
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    skip_role_assignment=False
)

print(f"\n✓ Role assignments completed successfully")

## Block 9: Deploy Network Security Perimeter

Deploy NSP with enforced mode to protect Storage Account.

**NSP Rules (Hardcoded - not configurable):**
- Inbound: Allow traffic from current subscription
- Inbound: Allow MicrosoftPublicIPSpace service tag
- Outbound: Allow all FQDNs (*)

**Note:** Only run this block if NSP is configured in config.py

In [ ]:
nsp_name = config.get('nsp_name', '')
nsp_profile_name = config.get('nsp_profile_name', '')
nsp_access_mode_storage = config.get('nsp_access_mode_for_storage', '')

if nsp_name and not nsp_name.startswith('__'):
    print("=" * 60)
    print("DEPLOYING: Network Security Perimeter")
    print("=" * 60)
    print(f"NSP Name: {nsp_name}")
    print(f"NSP Profile: {nsp_profile_name}")
    print(f"Access Mode (Storage): {nsp_access_mode_storage or 'Not configured'}")
    
    # Check if NSP exists and handle location mismatch
    should_deploy, effective_location, message = check_and_prepare_deployment(
        resource_name=nsp_name,
        resource_type="nsp",
        config_location=config['location'],
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        credential=credential
    )
    print(f"\n{message}")
    
    if should_deploy:
        # Load parameters from template (resolved from config.py)
        nsp_params = get_parameters_for_deployment("NSP", config)
        
        # Deploy NSP
        nsp_outputs = deploy_arm(
            template_file_path="templates/NSP.Template.json",
            deployment_name="deepwiki-nsp",
            parameters=nsp_params,
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group']
        )
        
        print(f"\n✓ NSP deployed successfully")
        
        # Associate NSP with Storage Account (only if access mode is configured)
        if nsp_access_mode_storage and nsp_access_mode_storage in ['Enforced', 'Learning']:
            print(f"\nAssociating NSP with Storage Account (Mode: {nsp_access_mode_storage})...")
            
            nsp_assoc_params = {
                "nspName": nsp_name,
                "nspProfileName": nsp_profile_name,
                "nspAccessMode": nsp_access_mode_storage,
                "resourceName": config['storage_account_name'],
                "resourceType": "storage"
            }
            
            nsp_assoc_outputs = deploy_arm(
                template_file_path="templates/NSP.Associations.Template.json",
                deployment_name="deepwiki-nsp-storage-assoc",
                parameters=nsp_assoc_params,
                subscription_id=config['subscription_id'],
                resource_group=config['resource_group']
            )
            
            print(f"✓ NSP associated with Storage Account (Mode: {nsp_access_mode_storage})")
        else:
            print(f"\n⚠ Skipping Storage Account NSP association")
            print(f"  Reason: nsp_access_mode_for_storage not set to 'Enforced' or 'Learning'")
    else:
        print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
        print(f"  Please resolve the issue above before continuing.")
else:
    print("=" * 60)
    print("SKIPPING: Network Security Perimeter")
    print("=" * 60)
    print("Reason: NSP not configured in config.py")
    print("To enable NSP, set 'nsp_name' and 'nsp_profile_name' in config.py")

## Deployment Summary

Print summary of all deployed resources.

In [ ]:
print("=" * 60)
print("DEPLOYMENT SUMMARY")
print("=" * 60)
print(f"\nResource Group: {config['resource_group']}")
print(f"Location: {config['location']}")
print(f"\nResources Deployed:")
print(f"  ✓ Managed Identity: {config['dri_copilot_identity_name']}")
print(f"  ✓ Storage Account: {config['storage_account_name']}")
print(f"  {'✓' if config.get('is_creating_open_ai_endpoint', False) else '○'} Azure OpenAI: {config['open_ai_resource_name']}")
print(f"  ✓ Application Insights: {config['application_insights_name']}")
print(f"  ✓ Log Analytics: {config['log_analytics_workspace_name']}")
print(f"  ✓ App Service Plan: {config['app_service_plan_name']}")
print(f"  ✓ App Service: {config['app_service_name']}")
print(f"  {'✓' if config.get('nsp_name', '') else '○'} Network Security Perimeter: {config.get('nsp_name', 'Not configured')}")
print(f"\nRole Assignments:")
print(f"  ✓ Cognitive Services OpenAI User")
print(f"  ✓ Storage Blob Data Contributor")
print(f"  ✓ Monitoring Metrics Publisher")
print(f"\n" + "=" * 60)
print("Next Steps:")
print("  1. Update container image in App Service when ready")
print("  2. Configure environment variables for your application")
print("  3. Run deploy_planning.ipynb for optional resources (AML, Search, Key Vault)")
print("=" * 60)